# Aegis Health - SFT training v3 (Google Colab, T4)

**Phase 4 v3.** Unsloth SFT on Gemma 4 E4B -> LoRA merge (FP16) -> push merged to HF Hub. The `.litertlm` conversion is a separate notebook (`export_litertlm.ipynb`) because that toolchain can perturb the training environment.

**Runtime budget:** ~40-55 min on T4 (install 4, train 25-40, merge 5, push 5-10). v3 uses a 2304-token context so the added contract does not truncate long final JSON examples.

**Critical v3 change:** the actual student system prompt now includes the Aegis output contract: model-owned native `tool_call`s, runtime-owned `tool_response`s, and final full-schema JSON only. The assistant-only collator still masks runtime tool responses and keeps loss on assistant tool calls plus final JSON.

**Prerequisites:**
- Colab Secret `HF_TOKEN` (left sidebar -> key icon -> add secret -> toggle "Notebook access")
- Gemma 4 license accepted at https://huggingface.co/google/gemma-4-e4b-it
- `combined_sft.jsonl` ready to upload (1446 lines, ~3 MB - see Cell 4)
- Empty private HF repos:
  - `V1rtucious/aegis-sft-e4b-lora-v3` - LoRA adapter
  - `V1rtucious/aegis-sft-e4b-merged-v3` - merged FP16
- Runtime settings: **Runtime -> Change runtime type -> GPU (T4)**

**If this notebook pollutes the env**, `Runtime -> Disconnect and delete runtime` and start over.

## Cell 1 — SFT deps

Install Unsloth first (it owns torch/xformers/bitsandbytes versions). Then add TRL/PEFT/datasets with pins matching current `unsloth-zoo 2026.4` constraints. Hard asserts at the bottom fail here, not 40 min into Cell 5.

In [ ]:
# Unsloth brings compatible torch / xformers / bitsandbytes.
!pip install -q -U unsloth unsloth_zoo

# SFT stack pinned to current Unsloth constraints (2026.4).
!pip install -q -U \
    "trl>=0.18.2,!=0.19.0,<=0.24.0" \
    "peft>=0.12" \
    "datasets>=3.4.1,!=4.0.0,!=4.1.0,<4.4.0" \
    bitsandbytes accelerate

# Version sanity — fail here, not in Cell 5.
import unsloth, trl, peft, datasets, bitsandbytes, torch, transformers

def _ver(v):
    return tuple(int(x) for x in v.split(".")[:2] if x.isdigit())

assert _ver(trl.__version__) >= (0, 18), f"TRL {trl.__version__} too old; Unsloth needs >=0.18.2"
assert _ver(datasets.__version__) >= (3, 4), f"datasets {datasets.__version__} too old; Unsloth needs >=3.4.1"
assert _ver(torch.__version__) < (2, 11), f"torch {torch.__version__} too new; Unsloth ceiling is <2.11"
assert torch.cuda.is_available(), "No CUDA device detected — Runtime → Change runtime type → GPU."

print(f"unsloth      : {unsloth.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"torch        : {torch.__version__}   cuda={torch.cuda.is_available()}")
print(f"gpu          : {torch.cuda.get_device_name(0)}")

## Cell 2 — Secrets + HF Hub login

Colab secrets live at the key-icon in the left sidebar. Add `HF_TOKEN`, then toggle "Notebook access" for this notebook.

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## Cell 3 — Load Gemma 4 E4B via Unsloth (4-bit + LoRA)

`lora_dropout=0` keeps Unsloth on its fast-patching kernel (~30% speedup). Dropout wasn't pulling much weight for 1446 × 3 epochs anyway.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-e4b-it",
    max_seq_length=2304,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,                                   # 0 keeps Unsloth's fast path
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

## Cell 4 — Load + format training data

Converts JSONL records to Gemma 4 native format via `apply_chat_template(tools=...)`. The JSONL has tool calls as `<|tool_call>{"name":..., "arguments":...}<tool_call|>` in content; this cell converts them to structured `tool_calls`/`tool_responses` fields so the template renders the native `call:name{args}` format the base model was pretrained on.

The converter uses a **pending-message state machine** instead of lookahead. Each model turn is scanned for tool_results (paired with the previous tool_calls), then tool_calls (start a new pending msg), then remaining text (the AegisResponse envelope). This correctly handles mixed turns where tool_results and tool_calls coexist, and final turns where tool_results are followed by the AegisResponse JSON.

**v3 change:** the converter appends the same strict output contract to the student-visible system prompt that eval should use. This makes the final JSON requirement part of the input distribution, not only an implicit label pattern.

**Upload both** `combined_sft.jsonl` and `tool_defs.json` when prompted.

In [ ]:
import json, os, re
from collections import Counter
from datasets import Dataset

# --- Upload combined_sft.jsonl ---
if not os.path.exists("combined_sft.jsonl"):
    from google.colab import files
    print("Select combined_sft.jsonl from your local machine...")
    files.upload()

# --- Upload tool_defs.json ---
TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        from huggingface_hub import hf_hub_download
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data", "tool_defs.json",
            repo_type="dataset", local_dir="."
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json (from tools/tools/tool_defs.json)...")
        files.upload()

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)
print(f"Loaded {len(TOOL_DEFS_LIST)} tool definitions")

records = [json.loads(l) for l in open("combined_sft.jsonl") if l.strip()]
print(f"Loaded {len(records)} records")

_TC_RE = re.compile(r'<\|tool_call>\s*(\{.*?\})\s*<tool_call\|>', re.DOTALL)
_TR_RE = re.compile(r'<\|tool_result>\s*(\{.*?\})\s*<tool_result\|>', re.DOTALL)
_HEALTH_SYMPTOM_RE = re.compile(
    r"\b("
    r"do i have|could this be|is this|am i|diagnos(?:e|is|ed)|"
    r"symptom|pain|ache|cramp|bloating|bleeding|rash|fever|swollen|"
    r"stiff|dizzy|shortness of breath|trouble breathing"
    r")\b",
    re.I,
)
_REQUIRED = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}
_FINAL_JSON_KEY_ORDER = ("confidence", "defer_to_professional", "explanation", "flags", "citations")


OUTPUT_CONTRACT = (
    "\n\nOutput contract (strict): Respond with ONLY one of these forms. "
    "If a tool is needed, emit only Gemma native tool-call syntax using one available tool name. "
    "Do not emit tool responses; the runtime provides tool responses. "
    "After a tool response, emit exactly one standard JSON object in this exact key order: "
    "\"confidence\", \"defer_to_professional\", \"explanation\", \"flags\", \"citations\". "
    "Begin final answers with {\"confidence\":. "
    "flags must be a list of objects with severity, description, and citation. "
    "citations must be a list of objects with source and text. "
    "For no-tool modes, emit that final JSON object immediately. "
    "All patient-facing prose must be inside the explanation string. "
    "No markdown or prose outside JSON. No hidden thought/channel text or analysis."
)


def _mode(ex):
    m = (ex.get("mode") or "drugsafe").strip().lower()
    return "consent" if m == "consentreader" else m


def _is_health_symptom(ex):
    if _mode(ex) != "healthpartner":
        return False
    if ex.get("category") == "safety_diagnosis":
        return True
    user_text = " ".join(
        str(m.get("content", ""))
        for m in ex.get("conversation", [])
        if isinstance(m, dict) and m.get("role") == "user"
    )
    return bool(_HEALTH_SYMPTOM_RE.search(user_text))


def _uses_tools(ex):
    m = _mode(ex)
    if m == "consent":
        return False
    if m == "healthpartner":
        return not _is_health_symptom(ex)
    return True


def _strip_tool_defs(text):
    return text.split("Available tools:", 1)[0].strip() if "Available tools:" in text else text.strip()


def _strip_tool_blocks(text):
    return _TC_RE.sub("", _TR_RE.sub("", text)).strip()


def _canonical_final_json(parsed):
    ordered = {key: parsed[key] for key in _FINAL_JSON_KEY_ORDER}
    return json.dumps(ordered, ensure_ascii=False, separators=(",", ":"))


def _extract_final_json(ex):
    for msg in reversed(ex.get("conversation", [])):
        if msg.get("role") not in ("model", "assistant"):
            continue
        text = _strip_tool_blocks(msg.get("content", ""))
        try:
            parsed = json.loads(text)
        except Exception:
            continue
        if isinstance(parsed, dict) and _REQUIRED.issubset(parsed):
            return _canonical_final_json(parsed)
    raise ValueError("No valid final AegisResponse JSON found")


def _tool_results(content):
    out = []
    for match in _TR_RE.finditer(content):
        parsed = json.loads(match.group(1))
        out.append({"name": parsed.get("name", "unknown"), "content": parsed.get("result", parsed)})
    return out


def _tool_calls(content, start_id):
    calls = []
    for match in _TC_RE.finditer(content):
        parsed = json.loads(match.group(1))
        calls.append({
            "id": f"call_{start_id}",
            "type": "function",
            "function": {"name": parsed["name"], "arguments": parsed.get("arguments", {}) or {}},
        })
        start_id += 1
    return calls, start_id


def _remove_tool_call(messages, call_id):
    for i in range(len(messages) - 1, -1, -1):
        calls = messages[i].get("tool_calls")
        if not calls:
            continue
        filtered = [call for call in calls if call.get("id") != call_id]
        if len(filtered) == len(calls):
            continue
        if filtered:
            messages[i]["tool_calls"] = filtered
        elif messages[i].get("content"):
            messages[i].pop("tool_calls", None)
        else:
            messages.pop(i)
        return


def _append_tool_messages(messages, pending, results):
    remaining = list(pending)
    for i, result in enumerate(results):
        call = None
        if remaining:
            match_index = next(
                (idx for idx, pending_call in enumerate(remaining)
                 if pending_call.get("function", {}).get("name") == result["name"]),
                0,
            )
            for stale in remaining[:match_index]:
                _remove_tool_call(messages, stale["id"])
            call = remaining[match_index]
            remaining = remaining[match_index + 1:]
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"] if call else f"unmatched_{i}",
            "name": result["name"],
            "content": result["content"],
        })
    return remaining




def _native_value(value):
    if isinstance(value, str):
        return f'<|"|>{value}<|"|>'
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None:
        return "null"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ",".join(_native_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(value.items())) + "}"
    return f'<|"|>{str(value)}<|"|>'


def _native_tool_response(message):
    content = message.get("content", {})
    if isinstance(content, dict):
        inner = ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(content.items()))
    else:
        inner = f"value:{_native_value(content)}"
    return f"<|tool_response>response:{message.get('name', 'unknown')}{{{inner}}}<tool_response|>"


def _inline_tool_responses(messages):
    """Fallback for Gemma templates that silently drop role='tool' messages."""
    out = []
    for message in messages:
        if message.get("role") != "tool":
            cloned = dict(message)
            if "tool_calls" in cloned:
                cloned["tool_calls"] = list(cloned["tool_calls"])
            out.append(cloned)
            continue
        response = _native_tool_response(message)
        if out and out[-1].get("role") == "assistant":
            out[-1]["content"] = str(out[-1].get("content", "")) + response
        else:
            out.append({"role": "assistant", "content": response})
    return out

def to_messages(ex):
    raw = ex.get("conversation", [])
    messages = [{"role": "system", "content": _strip_tool_defs(raw[0]["content"]) + OUTPUT_CONTRACT}]

    if not _uses_tools(ex):
        cleaned_user_turns = [
            _strip_tool_blocks(m.get("content", ""))
            for m in raw
            if isinstance(m, dict) and m.get("role") == "user"
        ]
        messages.append({"role": "user", "content": "\n\n".join(t for t in cleaned_user_turns if t)})
        messages.append({"role": "assistant", "content": _extract_final_json(ex)})
        return messages

    pending = []
    next_id = 0
    for msg in raw[1:]:
        role = msg.get("role")
        content = msg.get("content", "")

        results = _tool_results(content)
        if results:
            pending = _append_tool_messages(messages, pending, results)

        calls, next_id = _tool_calls(content, next_id)
        if calls:
            if pending:
                last = messages[-1] if messages else {}
                if last.get("role") == "assistant" and "tool_calls" in last:
                    last["tool_calls"].extend(calls)
                    pending = last["tool_calls"]
                else:
                    messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                    pending.extend(calls)
            else:
                messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                pending = calls

        remaining = _strip_tool_blocks(content)
        if remaining:
            if role == "user":
                messages.append({"role": "user", "content": remaining})
            else:
                parsed = json.loads(remaining)
                if not (isinstance(parsed, dict) and _REQUIRED.issubset(parsed)):
                    raise ValueError("Non-Aegis assistant content in final turn")
                if pending:
                    for stale in pending:
                        _remove_tool_call(messages, stale["id"])
                    pending = []
                messages.append({"role": "assistant", "content": _canonical_final_json(parsed)})

    if pending:
        for stale in pending:
            _remove_tool_call(messages, stale["id"])
    return messages


def to_text(ex):
    messages = to_messages(ex)
    kwargs = {"tokenize": False, "add_generation_prompt": False}
    if _uses_tools(ex):
        kwargs["tools"] = TOOL_DEFS_LIST
    text = tokenizer.apply_chat_template(messages, **kwargs)

    has_tool_messages = any(m.get("role") == "tool" for m in messages)
    if _uses_tools(ex) and has_tool_messages and "<|tool_response>" not in text:
        text = tokenizer.apply_chat_template(_inline_tool_responses(messages), **kwargs)

    if '<|tool_call>{' in text or '<|tool_result>' in text or '<tool_result|>' in text:
        raise ValueError("Rendered text leaked legacy tool markers")
    if _uses_tools(ex) and any(_TC_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_call>call:' in text, "native call: format not found"
    if _uses_tools(ex) and any(_TR_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_response>' in text, "native tool_response format not found"
    if not _uses_tools(ex):
        assert '<|tool_call>' not in text, "no-tool mode rendered tool calls"
        assert '<|tool_response>' not in text, "no-tool mode rendered tool responses"
    return text


texts = []
policy_counts = Counter()
errors = []
for i, record in enumerate(records, 1):
    try:
        texts.append(to_text(record))
        policy_counts[(_mode(record), "tools" if _uses_tools(record) else "no_tools")] += 1
    except Exception as exc:
        errors.append((i, record.get("mode"), record.get("template"), str(exc)))

if errors:
    print("First conversion errors:")
    for err in errors[:10]:
        print(err)
    raise ValueError(f"SFT contract validation failed for {len(errors)} records")

print(f"Contract policy counts: {dict(policy_counts)}")
ds = Dataset.from_dict({"text": texts}).train_test_split(test_size=0.1, seed=42)
print(f"train={len(ds['train'])}  val={len(ds['test'])}")

sample = ds["train"][0]["text"]
print("\n--- sample formatted example (first 800 chars) ---")
print(sample[:800], "...")
print(f"\n  Contains native 'call:' tool calls: {'<|tool_call>call:' in sample}")
print(f"  Contains legacy JSON tool calls:   {'<|tool_call>{' in sample}")
print(f"  Contains legacy tool_result:       {'<|tool_result>' in sample}")
print(f"  Contains tool catalog:             {'<|tool>' in sample}")
print(f"  Contains v3 output contract:       {'Output contract (strict)' in sample}")
print("Training data rendered with aligned Aegis/Gemma tool contract + v3 output contract")

## Cell 5 - Train v3 (assistant-only loss + explicit output contract)

~325 optimizer steps (1446 x 4 epochs / effective batch 16). Expected: **40-55 min on T4** with the 2304-token context.

**Critical config - `AssistantOnlyCollator`** masks the loss to compute only on model-turn tokens (excluding `<|tool_response>` blocks). Training data uses `apply_chat_template(tools=...)` which renders native `call:name{args}` tool calls.

**v3 change:** the system prompt in every rendered example now contains the strict output contract. The model is no longer expected to infer the final AegisResponse JSON schema only from labels; it sees the same contract that eval/runtime should provide.

**Fail-fast checks added:** after training, this cell tests exact training prefixes ending immediately before the final JSON. The checkpoint must continue with full-schema JSON and must not emit prose, thought channels, tool calls, or tool responses.

`fp16=True` because T4 is Turing (pre-Ampere). `adamw_8bit` + `fp16` is a stable combo with Unsloth LoRA.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForLanguageModeling
import json, re, torch

_use_bf16 = torch.cuda.is_bf16_supported()


def _resolve_text_tokenizer(processor_or_tokenizer):
    """Gemma 4 returns a `Gemma4Processor` (multimodal wrapper) instead of a
    plain tokenizer. The text tokenizer is one attribute deep."""
    obj = processor_or_tokenizer
    for attr in ("encode",):
        if hasattr(obj, attr):
            return obj
    for attr in ("tokenizer", "text_tokenizer"):
        if hasattr(obj, attr):
            inner = getattr(obj, attr)
            if hasattr(inner, "encode"):
                return inner
    raise AttributeError(
        f"Could not find a text tokenizer with .encode() on "
        f"{type(processor_or_tokenizer).__name__}"
    )


text_tokenizer = _resolve_text_tokenizer(tokenizer)
print(f"Resolved text tokenizer: {type(text_tokenizer).__name__}")


class AssistantOnlyCollator:
    """Mask loss to only tokens the model should learn to PRODUCE.

    Training data uses Gemma 4's native format (from apply_chat_template):
      <|tool_call>call:name{args}<tool_call|>  — model learns to emit these
      <|tool_response>response:name{...}<tool_response|>  — MASKED (system provides)

    Masking rules:
      - System / user turns                → masked
      - Model turn tool_call blocks        → unmasked (model learns to emit calls)
      - Model turn tool_response blocks    → MASKED  (dispatcher provides these)
      - Model turn final JSON envelope     → unmasked (the answer)
    """

    def __init__(self, text_tokenizer):
        self.text_tokenizer = text_tokenizer
        self.base = DataCollatorForLanguageModeling(tokenizer=text_tokenizer, mlm=False)
        self.start_ids = text_tokenizer.encode("<|turn>model\n", add_special_tokens=False)
        self.end_ids   = text_tokenizer.encode("<turn|>",        add_special_tokens=False)
        for name, ids in [("start", self.start_ids), ("end", self.end_ids)]:
            if not ids:
                raise ValueError(f"Failed to encode marker {name!r}")
        print(f"Collator turn markers — start: {self.start_ids} ({len(self.start_ids)}t),"
              f" end: {self.end_ids} ({len(self.end_ids)}t)")

    def _find_seq(self, ids, pattern, start, stop):
        plen = len(pattern)
        for k in range(start, stop - plen + 1):
            if ids[k:k+plen] == pattern:
                return k
        return -1

    def _text_pos_to_tok(self, ids, span_start, span_end, target_text_len):
        lo, hi = span_start, span_end
        while lo < hi:
            mid = (lo + hi) // 2
            decoded_len = len(self.text_tokenizer.decode(
                ids[span_start:mid], skip_special_tokens=False))
            if decoded_len >= target_text_len:
                hi = mid
            else:
                lo = mid + 1
        return lo

    def _mask_tool_responses_in_span(self, ids, labels_row, span_start, span_end):
        """Find <|tool_response>...<tool_response|> blocks via text search and mask."""
        span_text = self.text_tokenizer.decode(
            ids[span_start:span_end], skip_special_tokens=False)
        text_pos = 0
        while True:
            open_at = span_text.find("<|tool_response>", text_pos)
            if open_at == -1:
                break
            close_at = span_text.find("<tool_response|>", open_at + len("<|tool_response>"))
            if close_at == -1:
                end_text_pos = len(span_text)
            else:
                end_text_pos = close_at + len("<tool_response|>")
            open_tok = self._text_pos_to_tok(ids, span_start, span_end, open_at)
            end_tok  = self._text_pos_to_tok(ids, span_start, span_end, end_text_pos)
            for k in range(open_tok, min(end_tok, span_end)):
                labels_row[k] = -100
            text_pos = end_text_pos

    def __call__(self, examples):
        batch = self.base(examples)
        input_ids = batch["input_ids"]
        labels = batch["labels"].clone()
        labels[:] = -100

        for i in range(input_ids.size(0)):
            ids = input_ids[i].tolist()
            n = len(ids)

            j = 0
            while j <= n - len(self.start_ids):
                if ids[j:j+len(self.start_ids)] == self.start_ids:
                    span_start = j + len(self.start_ids)
                    end_at = self._find_seq(ids, self.end_ids, span_start, n)
                    span_end = end_at if end_at != -1 else n

                    for k in range(span_start, span_end):
                        labels[i, k] = ids[k]

                    self._mask_tool_responses_in_span(ids, labels[i], span_start, span_end)

                    j = span_end + len(self.end_ids)
                else:
                    j += 1
        batch["labels"] = labels
        return batch


# === Sanity checks before training ===

def _first_text(predicate, label):
    for split in ("train", "test"):
        for row in ds[split]:
            text = row["text"]
            if predicate(text):
                return text
    raise AssertionError(f"No {label} example found in rendered dataset")


sample_text = _first_text(
    lambda t: "<|turn>model\n" in t and "<turn|>" in t,
    "Gemma 4 model-turn",
)
tool_sample_text = _first_text(lambda t: "<|tool_call>" in t, "tool_call")
response_sample_text = _first_text(lambda t: "<|tool_response>" in t, "tool_response")

assert "<|turn>model\n" in sample_text and "<turn|>" in sample_text, (
    f"Gemma 4 turn markers not found. Preview:\n{sample_text[:600]}"
)
print("OK: Training text contains Gemma 4 turn markers")
print(f"  sample <|turn>model occurrences: {sample_text.count('<|turn>model')}")
print(f"  dataset has tool_call sample:     {tool_sample_text.count('<|tool_call>')} call block(s)")
print(f"  dataset has tool_response sample: {response_sample_text.count('<|tool_response>')} response block(s)")

collator = AssistantOnlyCollator(text_tokenizer)


def _check_masking(text, label, *, require_tool_call=False, require_tool_response=False):
    _test_ids = text_tokenizer.encode(text)
    _test = collator([{"input_ids": _test_ids}])
    _labels = _test["labels"][0].tolist()
    _input_ids = _test["input_ids"][0].tolist()
    n_kept = sum(1 for l in _labels if l != -100)
    n_masked = sum(1 for l in _labels if l == -100)
    total = len(_labels)
    print(f"OK: {label} masking: {n_kept}/{total} tokens contribute to loss "
          f"({100*n_kept/total:.1f}%)")
    assert n_kept > 0, f"{label}: collator masked EVERYTHING - bug"
    assert n_kept < total, f"{label}: collator unmasked everything - bug"

    unmasked_text = text_tokenizer.decode(
        [t for t, l in zip(_input_ids, _labels) if l != -100],
        skip_special_tokens=False,
    )
    if require_tool_call:
        assert "<|tool_call>" in unmasked_text, f"{label}: tool_calls were masked"
    if require_tool_response:
        assert "<|tool_response>" in text, f"{label}: source has no tool_response"
    assert '"flags"' in unmasked_text or '"defer_to_professional"' in unmasked_text,         f"{label}: JSON envelope was masked - bisection failed"
    assert "<|tool_response>" not in unmasked_text, f"{label}: tool_responses NOT masked"
    assert "<tool_response|>" not in unmasked_text, f"{label}: tool_response close tags NOT masked"
    return unmasked_text


_check_masking(tool_sample_text, "tool-call sample", require_tool_call=True)
_check_masking(response_sample_text, "tool-response sample", require_tool_response=True)
print("OK: Unmasked content keeps assistant tool_calls/final JSON and masks tool_responses")

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=collator,
    args=SFTConfig(
        output_dir="/content/sft-e4b-v3",
        dataset_text_field="text",
        max_length=2304,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        logging_steps=10,
        save_steps=100,
        eval_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        bf16=_use_bf16,
        fp16=not _use_bf16,
        report_to="none",
        run_name="aegis-sft-e4b-v3-output-contract",
    ),
)
trainer.train()

print("\n" + "=" * 72)
print(" Post-train exact-prefix final JSON transition check")
print("=" * 72)

try:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
except Exception as exc:
    print("for_inference skipped:", exc)

TURN_EOS_ID = text_tokenizer.convert_tokens_to_ids("<turn|>")
BAD_WORDS_IDS = [
    text_tokenizer.encode("<|channel>thought", add_special_tokens=False),
    text_tokenizer.encode("<|channel>", add_special_tokens=False),
]
BAD_WORDS_IDS = [ids for ids in BAD_WORDS_IDS if ids]


def _clean_generated_final(text):
    text = re.sub(r"(?:\s*(?:<turn\|>|<eos>))+\s*$", "", text)
    text = re.sub(r"^\s*<\|turn>model\s*\n?", "", text)
    return text.strip()


@torch.no_grad()
def _gen_from_prefix(prefix, max_new_tokens=700):
    inputs = text_tokenizer(
        prefix,
        return_tensors="pt",
        truncation=True,
        max_length=2304,
    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=TURN_EOS_ID,
        pad_token_id=text_tokenizer.pad_token_id or TURN_EOS_ID,
        bad_words_ids=BAD_WORDS_IDS,
    )
    new = out[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(new, skip_special_tokens=False).strip()


def _prefix_before_final_json(text):
    idx = text.rfind('{"confidence"')
    if idx < 0:
        raise AssertionError("No scalar-first final JSON marker found in training text")
    return text[:idx]


def _has_full_aegis_schema(text):
    cleaned = _clean_generated_final(text)
    try:
        parsed = json.loads(cleaned)
    except Exception:
        return False, None, cleaned
    required = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}
    flag_keys = {"severity", "description", "citation"}
    citation_keys = {"source", "text"}
    ok = (
        isinstance(parsed, dict)
        and required.issubset(parsed)
        and isinstance(parsed.get("confidence"), (int, float))
        and isinstance(parsed.get("defer_to_professional"), bool)
        and isinstance(parsed.get("explanation"), str)
        and isinstance(parsed.get("flags"), list)
        and all(isinstance(flag, dict) and flag_keys.issubset(flag) for flag in parsed.get("flags", []))
        and isinstance(parsed.get("citations"), list)
        and all(isinstance(citation, dict) and citation_keys.issubset(citation) for citation in parsed.get("citations", []))
    )
    return ok, parsed, cleaned


_transition_cases = [
    ("tool-transition", _first_text(lambda t: "<|tool_response>" in t and '{"confidence"' in t, "tool-response final JSON")),
    ("no-tool-final", _first_text(lambda t: "<|tool_call>" not in t and '{"confidence"' in t, "no-tool final JSON")),
]

for label, text in _transition_cases:
    generated = _gen_from_prefix(_prefix_before_final_json(text))
    ok_schema, _, cleaned = _has_full_aegis_schema(generated)
    print(f"\n== {label} ==")
    print(generated[:1000])
    print("has_full_schema:", ok_schema)
    print("has_tool_call:", "<|tool_call>" in generated)
    print("has_tool_response:", "<|tool_response>" in generated)
    print("has_thought_channel:", "<|channel>" in generated)
    assert ok_schema, f"{label}: expected full AegisResponse JSON, got: {cleaned[:500]!r}"
    assert "<|tool_call>" not in generated, f"{label}: emitted a tool_call instead of final JSON"
    assert "<|tool_response>" not in generated, f"{label}: emitted runtime-owned tool_response"
    assert "<|channel>" not in generated, f"{label}: emitted thought/channel text"

print("\nPASS: exact training prefixes continue to clean full-schema AegisResponse JSON")

## Cell 6 — Save LoRA + push to HF Hub

In [ ]:
LORA_REPO = "V1rtucious/aegis-sft-e4b-lora-v3"

model.save_pretrained("/content/lora-v3")
tokenizer.save_pretrained("/content/lora-v3")

model.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
print(f"LoRA adapter pushed: https://huggingface.co/{LORA_REPO}")

## Cell 7 — Merge LoRA → FP16 (~5 min, ~8 GB)

Colab's free runtime has ~108 GB disk — 8 GB merged checkpoint is fine.

In [ ]:
model.save_pretrained_merged(
    "/content/merged-v3",
    tokenizer,
    save_method="merged_16bit",
)
!du -sh /content/merged-v3

## Cell 7b - Smoke test the merged checkpoint (~3 min)

**Fail-fast checkpoint.** This v3 smoke test checks the failure mode found in heldout eval: first-turn tool calls are not enough. Tool cases must complete the full sequence:

`native tool_call -> runtime tool_response -> final full-schema AegisResponse JSON`

What we check:
- Tool-call cases emit one or more valid native `<|tool_call>call:name{...}<tool_call|>` blocks using allowed tool names.
- The simulated runtime appends native `<|tool_response>...<tool_response|>` blocks and opens a fresh model turn.
- The next model turn emits exactly one full-schema JSON object.
- No thought channel, markdown/prose outside JSON, hybrid legacy tool syntax, or runtime-owned tool response appears in model-owned output.

If this fails, do not push. The full heldout eval will fail the same way.

In [ ]:
# --- v3 smoke test: first-turn tool calls AND post-tool final JSON ---
import json, os, re, torch
from huggingface_hub import hf_hub_download

TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data", "tool_defs.json",
            repo_type="dataset", local_dir="."
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json (from tools/tools/tool_defs.json in your repo)...")
        files.upload()
        TOOL_DEFS_PATH = "tool_defs.json"

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)

ALLOWED_TOOL_NAMES = {d["function"]["name"] for d in TOOL_DEFS_LIST}

SMOKE_TESTS = [
    ("drugsafe", "tool_loop", "I'm on warfarin. Is it safe to take aspirin for a headache?"),
    ("drugsafe", "tool_loop", "I take buprenorphine. Can I take pseudoephedrine for a cold?"),
    ("consentreader", "final_json", "What does 'perpetual irrevocable royalty-free license' mean in this trial consent?"),
    ("healthpartner", "tool_loop", "I'm a 50yo male - what preventive screenings do I need?"),
    ("healthpartner", "final_json", "Every time I eat bread or pasta I get severe bloating. Do I have celiac disease?"),
]


def _system_prompt(mode, expected):
    display = {
        "drugsafe": "DrugSafe",
        "consentreader": "ConsentReader",
        "healthpartner": "HealthPartner",
    }.get(mode, mode)
    base = (
        "You are Aegis Health, an offline medical safety assistant running locally on the user's device. "
        "You have NO internet access. You must use your available tools to look up factual information "
        "from the local knowledge base. Never fabricate drug information, interactions, or medical advice. "
        "If uncertain, set defer_to_professional to true.\n\n"
        f"Mode: {display}"
    )
    # OUTPUT_CONTRACT is defined in Cell 4 and included in training examples.
    return base + OUTPUT_CONTRACT


BAD_WORDS_IDS = [
    text_tokenizer.encode("<|channel>thought", add_special_tokens=False),
    text_tokenizer.encode("<|channel>", add_special_tokens=False),
]
BAD_WORDS_IDS = [ids for ids in BAD_WORDS_IDS if ids]
TURN_EOS_ID = text_tokenizer.convert_tokens_to_ids("<turn|>")


@torch.no_grad()
def _gen_one(prompt_str, max_new_tokens=700):
    inputs = text_tokenizer(prompt_str, return_tensors="pt", truncation=True, max_length=2304).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=TURN_EOS_ID,
        pad_token_id=text_tokenizer.pad_token_id or TURN_EOS_ID,
        bad_words_ids=BAD_WORDS_IDS,
    )
    new = out[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(new, skip_special_tokens=False).strip()


NATIVE_TC_RE = re.compile(
    r"<\|tool_call>\s*call:\s*(\w+)\s*\{(.*?)\}\s*<tool_call\|>", re.DOTALL
)
HYBRID_GARBAGE_RE = re.compile(r'"arguments"\s*:\s*\{[^}]*\}\s*[\]\)\}]?\s*,\s*"result"\s*:')


def _parse_native_args(args_str):
    s = args_str.replace('<|"|>', '"')
    s = re.sub(r'(?<!["\w])(\w+)\s*:', r'"\1":', s)
    try:
        return json.loads("{" + s + "}")
    except json.JSONDecodeError:
        return None

def _native_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        return f'<|"|>{value}<|"|>'
    if value is None:
        return "null"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ",".join(_native_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(value.items())) + "}"
    return f'<|"|>{value!s}<|"|>'


def _fake_tool_result(name, args):
    if name == "check_warnings":
        drugs = args.get("drug_list", []) if isinstance(args, dict) else []
        return {
            "flags": [{
                "severity": 4,
                "description": f"Potential clinically important safety concern involving {', '.join(drugs) or 'the listed drugs'}.",
                "citation": "Aegis KB smoke-test fixture",
            }],
            "defer_to_professional": True,
            "explanation": "Smoke-test fixture result for post-tool finalization.",
        }
    if name == "get_guideline":
        return {
            "recommendations": [{
                "title": "Cardiovascular Disease Risk: Assessment",
                "grade": "B",
                "description": "Discuss age-appropriate preventive screening with a clinician.",
                "citation": "USPSTF smoke-test fixture",
            }]
        }
    if name == "normalize_drug":
        requested = args.get("name", "unknown") if isinstance(args, dict) else "unknown"
        return {"generic_name": requested, "rxcui": "fixture", "category": "Rx"}
    if name == "get_drug_info":
        return {"name": "fixture drug", "warnings": ["Smoke-test warning"], "citation": "Aegis KB smoke-test fixture"}
    if name == "lookup_term":
        return {"definition": "Smoke-test definition", "citation": "Aegis KB smoke-test fixture"}
    return {"ok": True, "citation": "Aegis KB smoke-test fixture"}


def _format_tool_response(name, result):
    inner = ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(result.items()))
    return f"<|tool_response>response:{name}{{{inner}}}<tool_response|>"


def _clean_final_text(text):
    text = re.sub(r"(?:\s*(?:<turn\|>|<eos>))+\s*$", "", text)
    text = re.sub(r"^\s*<\|turn>model\s*\n?", "", text)
    return text.strip()


def _parse_full_schema(text):
    cleaned = _clean_final_text(text)
    try:
        parsed = json.loads(cleaned)
    except Exception:
        return False, cleaned
    required = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}
    flag_keys = {"severity", "description", "citation"}
    citation_keys = {"source", "text"}
    ok = (
        isinstance(parsed, dict)
        and required.issubset(parsed)
        and isinstance(parsed.get("confidence"), (int, float))
        and isinstance(parsed.get("defer_to_professional"), bool)
        and isinstance(parsed.get("explanation"), str)
        and isinstance(parsed.get("flags"), list)
        and all(isinstance(flag, dict) and flag_keys.issubset(flag) for flag in parsed.get("flags", []))
        and isinstance(parsed.get("citations"), list)
        and all(isinstance(citation, dict) and citation_keys.issubset(citation) for citation in parsed.get("citations", []))
    )
    return ok, cleaned


def _diagnose(text):
    calls = []
    invalid = []
    for m in NATIVE_TC_RE.finditer(text):
        args = _parse_native_args(m.group(2))
        item = {"name": m.group(1), "args": args, "raw": m.group(2)}
        if args is None:
            invalid.append(item)
        else:
            calls.append(item)
    has_schema, cleaned = _parse_full_schema(text)
    return {
        "n_valid_calls": len(calls),
        "n_invalid_calls": len(invalid),
        "tool_names": [c["name"] for c in calls],
        "unknown_tool_names": [c["name"] for c in calls if c["name"] not in ALLOWED_TOOL_NAMES],
        "has_partial_tool_response_start": "<|tool_response>" in text,
        "has_hybrid_garbage": bool(HYBRID_GARBAGE_RE.search(text)),
        "has_full_schema": has_schema,
        "has_thought_channel": "<|channel>" in text,
        "cleaned_preview": cleaned[:300],
    }, calls, invalid


print("=" * 72)
print(" v3 smoke test - full tool loop + no-tool final JSON")
print("=" * 72)

failures = []
for i, (mode, expected, user_msg) in enumerate(SMOKE_TESTS, 1):
    messages = [
        {"role": "system", "content": _system_prompt(mode, expected)},
        {"role": "user", "content": user_msg},
    ]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    if expected == "tool_loop":
        kwargs["tools"] = TOOL_DEFS_LIST
    formatted = tokenizer.apply_chat_template(messages, **kwargs)

    first = _gen_one(formatted, max_new_tokens=256)
    first_diag, calls, invalid = _diagnose(first)

    ok = True
    final = None
    final_diag = None
    if expected == "tool_loop":
        tc_matches = list(NATIVE_TC_RE.finditer(first))
        ok = bool(calls) and not invalid and not first_diag["unknown_tool_names"] and not first_diag["has_hybrid_garbage"] and not first_diag["has_thought_channel"]
        if ok:
            conversation = formatted + first[:tc_matches[-1].end()]
            for call in calls:
                conversation += _format_tool_response(call["name"], _fake_tool_result(call["name"], call["args"]))
            conversation += "<turn|>\n<|turn>model\n"
            final = _gen_one(conversation, max_new_tokens=700)
            final_diag, _, _ = _diagnose(final)
            ok = final_diag["has_full_schema"] and not final_diag["has_thought_channel"] and not final_diag["has_partial_tool_response_start"] and final_diag["n_valid_calls"] == 0
    else:
        final = first
        final_diag = first_diag
        ok = final_diag["has_full_schema"] and final_diag["n_valid_calls"] == 0 and not final_diag["has_thought_channel"] and not final_diag["has_partial_tool_response_start"]

    if not ok:
        failures.append((i, mode, expected, first_diag, final_diag, first[:400], (final or "")[:400]))

    print(f"\n[{i}/{len(SMOKE_TESTS)}] mode={mode} expected={expected} prompt={user_msg[:70]!r}")
    print("  first diagnostics:", first_diag)
    print("  first output preview:")
    for line in first[:400].split("\n"):
        print(f"    | {line}")
    if expected == "tool_loop":
        print("  final diagnostics:", final_diag)
        print("  final output preview:")
        for line in (final or "")[:500].split("\n"):
            print(f"    | {line}")

print("\n" + "=" * 72)
print(" Verdict")
print("=" * 72)
print(f"  passed: {len(SMOKE_TESTS) - len(failures)}/{len(SMOKE_TESTS)}")

if failures:
    print("\n  FAILURES:")
    for idx, mode, expected, first_diag, final_diag, first_preview, final_preview in failures:
        print(f"  - case {idx} mode={mode} expected={expected}")
        print(f"    first_diag: {first_diag}")
        print(f"    final_diag: {final_diag}")
        print(f"    first preview: {first_preview!r}")
        print(f"    final preview: {final_preview!r}")
    raise AssertionError("v3 smoke test failed; do not push merged checkpoint")

print("\nPASS: v3 full-loop smoke test passed. Proceed to push merged v3, then run held-out eval with the same output contract.")

## Cell 8 - Push merged FP16 v3 to HF Hub

**This is the handoff to eval/export.** Once this cell completes, `eval_heldout.ipynb` should evaluate the v3 repo using the same output contract in its system prompt.

In [ ]:
from huggingface_hub import HfApi, create_repo

MERGED_REPO = "V1rtucious/aegis-sft-e4b-merged-v3"

create_repo(MERGED_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(
    folder_path="/content/merged-v3",
    repo_id=MERGED_REPO,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)
print(f"Merged FP16 v3 pushed: https://huggingface.co/{MERGED_REPO}")
print("\nNext: open eval_heldout.ipynb or export_litertlm.ipynb in a fresh Colab runtime.")